In [1]:
from pathlib import Path
import hashlib
import json
import os
import shutil
import subprocess
import sys
import zipfile
from html.parser import HTMLParser

JUPYTER_ROOT = Path("/home/xilinx/jupyter_notebooks")
ZIP_PATH = JUPYTER_ROOT / "spectrum_software_v1_r4_bplus_board.zip"
TARGET = JUPYTER_ROOT / "spectrum_software_v1_r4_bplus"

EXPECTED_ZIP_SHA256 = (
    "0A83727DB5B5355D6854001EED3F64B37EC25BBD4022ED3B6D54F25262ACCC9C"
)
EXPECTED_MANIFEST_SHA256 = (
    "827F7446E686E9B61695533ECA96FF89C653739928A638597891954A671043AE"
)
EXPECTED_BUILD_REVISION = "spectrum_software_1_r4_bplus"
EXPECTED_MANIFEST_FILES = 553

# 用户明确要求不留备份。
# 只删除 TARGET 软件目录，不删除 ../user_data。
if not ZIP_PATH.is_file():
    raise FileNotFoundError(
        f"没有找到 {ZIP_PATH}。请先上传最新 R4 board ZIP。"
    )

# 先验证 ZIP，再删除旧目录。
actual_zip_hash = hashlib.sha256(
    ZIP_PATH.read_bytes()
).hexdigest().upper()

print("ZIP SHA256:", actual_zip_hash)

if actual_zip_hash != EXPECTED_ZIP_SHA256:
    raise RuntimeError("ZIP SHA256 不匹配，停止操作。")

with zipfile.ZipFile(ZIP_PATH) as archive:
    bad_member = archive.testzip()

    if bad_member is not None:
        raise RuntimeError(f"ZIP CRC 错误: {bad_member}")

    files = [
        item for item in archive.infolist()
        if not item.is_dir()
    ]
    all_names = [
        item.filename for item in archive.infolist()
    ]

    if not files:
        raise RuntimeError("ZIP 中没有文件。")

    root_prefix = "spectrum_software_v1_r4_bplus/"

    if not all(
        item.filename.startswith(root_prefix)
        for item in files
    ):
        raise RuntimeError(
            "ZIP 顶层目录不是 spectrum_software_v1_r4_bplus/。"
        )

    if any(
        "user_records" in name or "/user_data/" in name
        for name in all_names
    ):
        raise RuntimeError(
            "发布 ZIP 不应包含用户数据或用户记录目录。"
        )

    jupyter_root = JUPYTER_ROOT.resolve()

    for item in files:
        relative = Path(item.filename)

        if relative.is_absolute() or ".." in relative.parts:
            raise RuntimeError(
                f"不安全的 ZIP 路径: {item.filename}"
            )

        destination = (JUPYTER_ROOT / relative).resolve()

        if not str(destination).startswith(
            str(jupyter_root) + os.sep
        ):
            raise RuntimeError(
                f"ZIP 路径越界: {item.filename}"
            )

# 严格确认删除目标。
expected_target = (
    JUPYTER_ROOT / "spectrum_software_v1_r4_bplus"
).absolute()

if TARGET.absolute() != expected_target:
    raise RuntimeError("R4 目标路径校验失败。")

if TARGET.is_symlink():
    raise RuntimeError("R4 目标不能是符号链接。")

# 用户明确要求不留旧版本备份。
if TARGET.exists():
    if not TARGET.is_dir():
        raise RuntimeError(f"{TARGET} 不是目录，停止操作。")

    print("删除旧 R4 软件目录:", TARGET)
    shutil.rmtree(TARGET)
else:
    print("没有旧 R4 软件目录。")

# 解压最新版本。
with zipfile.ZipFile(ZIP_PATH) as archive:
    archive.extractall(JUPYTER_ROOT)

print("解压完成:", TARGET)

# 检查关键文件。
required_files = (
    "analysis_recorder.py",
    "record_store.py",
    "response_recorder.py",
    "user_analysis_source.py",
    "bplus/web/replay.js",
    "spectrum_web/workspace.js",
    "software_manifest.json",
)

for relative in required_files:
    path = TARGET / relative

    if not path.is_file():
        raise FileNotFoundError(
            f"R4 缺少关键文件: {relative}"
        )

# 校验 manifest。
manifest_path = TARGET / "software_manifest.json"
actual_manifest_hash = hashlib.sha256(
    manifest_path.read_bytes()
).hexdigest().upper()

print("Manifest SHA256:", actual_manifest_hash)

if actual_manifest_hash != EXPECTED_MANIFEST_SHA256:
    raise RuntimeError(
        "software_manifest.json SHA256 不匹配。"
    )

manifest = json.loads(
    manifest_path.read_text(encoding="utf-8")
)

if len(manifest["files_sha256"]) != EXPECTED_MANIFEST_FILES:
    raise RuntimeError(
        "manifest 文件数量不正确："
        f'{len(manifest["files_sha256"])}'
    )

# 独立进程执行完整性校验。
verification_code = f"""
import json
from spectrum_integrity import verify_software

result = verify_software('.')
print(json.dumps(result, indent=2))

if result.get('build_revision') != {EXPECTED_BUILD_REVISION!r}:
    raise SystemExit('build_revision mismatch')

if result.get('verified_files') != {EXPECTED_MANIFEST_FILES}:
    raise SystemExit('verified file count mismatch')
"""

completed = subprocess.run(
    [sys.executable, "-c", verification_code],
    cwd=TARGET,
    text=True,
    capture_output=True,
    check=True,
)

print(completed.stdout)

# 检查页面可见入口和内部标签隔离。
class VisibleTextParser(HTMLParser):
    def __init__(self):
        super().__init__()
        self.parts = []

    def handle_data(self, data):
        text = data.strip()

        if text:
            self.parts.append(text)


def visible_text(path):
    parser = VisibleTextParser()
    parser.feed(path.read_text(encoding="utf-8"))
    return " ".join(parser.parts)


root_text = visible_text(
    TARGET / "spectrum_web" / "index.html"
)
demo_text = visible_text(
    TARGET / "bplus" / "web" / "index.html"
)

for forbidden in (
    "B-X",
    "M4 / RC",
    "B 拓展",
    "M4 双声道",
):
    if forbidden in root_text or forbidden in demo_text:
        raise RuntimeError(
            f"最终页面仍包含内部标签: {forbidden}"
        )

for label in (
    "当前频谱",
    "深入分析",
    "我的记录",
    "演示样例",
    "频率响应",
):
    if label not in root_text:
        raise RuntimeError(
            f"最终页面缺少入口: {label}"
        )

if "请选择实验记录（可选）" not in demo_text:
    raise RuntimeError(
        "演示样例页面缺少按需加载的空白默认状态。"
    )

# 用户记录位于软件目录之外，本次删除不会影响。
USER_RECORDS = JUPYTER_ROOT / "user_data" / "records"
USER_RECORDS.mkdir(parents=True, exist_ok=True)

print("页面结构、用户数据隔离和关键文件检查通过。")
print("用户记录目录:", USER_RECORDS)
print("R4 已无备份覆盖安装完成。")

ZIP SHA256: 0A83727DB5B5355D6854001EED3F64B37EC25BBD4022ED3B6D54F25262ACCC9C
没有旧 R4 软件目录。
解压完成: /home/xilinx/jupyter_notebooks/spectrum_software_v1_r4_bplus
Manifest SHA256: 827F7446E686E9B61695533ECA96FF89C653739928A638597891954A671043AE
{
  "verified": true,
  "software_revision": "spectrum_software_1",
  "build_revision": "spectrum_software_1_r4_bplus",
  "manifest_sha256": "827F7446E686E9B61695533ECA96FF89C653739928A638597891954A671043AE",
  "verified_files": 553
}

页面结构、用户数据隔离和关键文件检查通过。
用户记录目录: /home/xilinx/jupyter_notebooks/user_data/records
R4 已无备份覆盖安装完成。


In [ ]:
cd /home/xilinx/jupyter_notebooks/spectrum_software_v1_r4_bplus
python3 spectrum_app.py --mode live --bind 0.0.0.0 --port 8091